# Chapter 5: Beyond Binary Classification

> Every complex prediction problem — imbalanced, multiclass, ranked — can be reduced to the binary classifier you already know how to build.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 4 (Practical Issues) &nbsp;|&nbsp; **Time:** ~45 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 5

---

## Learning Objectives

- Represent imbalanced classification as an alpha-weighted binary classification problem
- Implement subsampling as a reduction from weighted to plain binary classification
- Implement **One-versus-All (OVA)** and **All-versus-All (AVA)** reductions from multiclass to binary classification
- Understand the trade-off in training cost and robustness between OVA and AVA

## The Problem

Binary classifiers are the "black box" you already know how to build (decision trees, KNN, perceptron). But most real prediction problems aren't plain binary classification: fraud detection has 1000x more legitimate transactions than fraudulent ones (**imbalanced data**), and document categorization has more than 2 possible topics (**multiclass**).

Rather than inventing new algorithms from scratch, this chapter shows how to **reduce** these harder problems down to ordinary binary classification, reusing everything you already have.

## The Concept

**Two families of reductions:**

```
Imbalanced binary problem
(alpha times more costly to miss positives)
      │
      ▼
Subsample negatives
(keep positives, keep 1/alpha of negatives)
      │
      ▼
Plain binary classifier


Multiclass problem (K classes)
      │
      ├──► One-versus-All: train K binary classifiers (class i vs. rest)
      │        │
      │        ▼
      │    Predict: argmax of the K scores
      │
      └──► All-versus-All: train K(K-1)/2 binary classifiers (class i vs. class j)
               │
               ▼
           Predict: class with the most pairwise wins
```

### Key Ideas

- **Reductions preserve guarantees:** if your binary classifier achieves error `e`, the theory says the weighted predictor achieves error `alpha * e` (Theorem 2) — the reduction doesn't lose more than that factor.
- **Subsampling can go too far:** for large `alpha`, you throw away almost all of the negative examples, which can leave too little data to learn from if the dataset is already small — the theory's guarantee is about the *rate*, not about having enough absolute data.
- **OVA trains K classifiers, AVA trains K(K-1)/2:** AVA scales quadratically in the number of classes, but each of its sub-problems is "easier" (only involves 2 classes' worth of data), which can make it more accurate in practice, at higher computational cost (Theorem 3 vs. Theorem 4).
- **OVA can be "brittle":** since ties are broken by whichever binary classifier's score is highest, a single confidently-wrong classifier can override several correct ones.

## Build It

### Setup

We reuse the Digits and Breast Cancer Wisconsin datasets from earlier chapters, plus scikit-learn's `OneVsRestClassifier` / `OneVsOneClassifier` and its own `Perceptron` for comparison against our from-scratch multiclass reductions.

In [1]:
import numpy as np
from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron as SKPerceptron
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.metrics import accuracy_score, f1_score

RNG = np.random.RandomState(3)

### Step 1: A Simple Binary Base Learner

All the reductions in this chapter are built on top of *some* binary classifier — here we reuse the vanilla perceptron from Chapter 3 (expecting ±1 labels) as that reusable building block.

In [2]:
class SimplePerceptron:
    def __init__(self, max_iter=30, random_state=0):
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        N, D = X.shape
        self.w = np.zeros(D)
        self.b = 0.0
        rng = np.random.RandomState(self.random_state)
        order = np.arange(N)
        for _ in range(self.max_iter):
            rng.shuffle(order)
            for n in order:
                a = self.w @ X[n] + self.b
                if y[n] * a <= 0:
                    self.w += y[n] * X[n]
                    self.b += y[n]
        return self

    def decision_function(self, X):
        return np.asarray(X) @ self.w + self.b

    def predict(self, X):
        return np.sign(self.decision_function(X))

### Step 2: Subsampling for Alpha-Weighted Classification (Section 5.1, Algorithm 11)

`SubsampleMap` turns an alpha-weighted binary problem into a plain (unweighted) one: keep every positive example, but keep each negative example only with probability `1/alpha`. Train an ordinary binary classifier on the result.

In [3]:
def subsample_map(X, y, alpha, random_state=0):
    rng = np.random.RandomState(random_state)
    X, y = np.asarray(X), np.asarray(y)
    keep = np.ones(len(y), dtype=bool)
    neg_idx = np.where(y == -1)[0]
    u = rng.uniform(0, 1, size=len(neg_idx))
    drop = neg_idx[u >= 1.0 / alpha]
    keep[drop] = False
    return X[keep], y[keep]

### Step 3: One-versus-All (Algorithm 12: Train / Algorithm 13: Test)

OVA trains one binary classifier per class, each one distinguishing "this class" from "everything else." At prediction time, every classifier's raw decision score is computed, and the class whose classifier is most confident wins.

In [4]:
class OVAClassifier:
    def __init__(self, base_learner_fn, classes=None):
        self.base_learner_fn = base_learner_fn
        self.classes = classes

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        self.classes_ = self.classes if self.classes is not None else np.unique(y)
        self.models_ = {}
        for c in self.classes_:
            y_bin = np.where(y == c, 1, -1)
            model = self.base_learner_fn()
            model.fit(X, y_bin)
            self.models_[c] = model
        return self

    def predict(self, X):
        X = np.asarray(X)
        scores = np.column_stack(
            [self.models_[c].decision_function(X) for c in self.classes_]
        )
        return self.classes_[np.argmax(scores, axis=1)]

### Step 4: All-versus-All (Algorithm 14: Train / Algorithm 15: Test)

AVA trains one binary classifier per **pair** of classes, using only the examples belonging to those two classes. At prediction time, every pairwise classifier casts a vote for whichever of its two classes it prefers, and the class with the most votes wins.

In [5]:
class AVAClassifier:
    def __init__(self, base_learner_fn, classes=None):
        self.base_learner_fn = base_learner_fn
        self.classes = classes

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        self.classes_ = self.classes if self.classes is not None else np.unique(y)
        self.models_ = {}
        for i, ci in enumerate(self.classes_):
            for cj in self.classes_[i + 1:]:
                mask = (y == ci) | (y == cj)
                y_bin = np.where(y[mask] == ci, 1, -1)
                model = self.base_learner_fn()
                model.fit(X[mask], y_bin)
                self.models_[(ci, cj)] = model
        return self

    def predict(self, X):
        X = np.asarray(X)
        N = X.shape[0]
        scores = {c: np.zeros(N) for c in self.classes_}
        for (ci, cj), model in self.models_.items():
            pred = np.sign(model.decision_function(X))
            scores[ci] += (pred == 1)
            scores[cj] += (pred == -1)
        score_matrix = np.column_stack([scores[c] for c in self.classes_])
        return self.classes_[np.argmax(score_matrix, axis=1)]


def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

## Use It — Real Data

### Experiment A: Subsampling for Imbalanced Data (Section 5.1, Theorem 2)

We artificially imbalance the Breast Cancer Wisconsin dataset by keeping only 8% of the malignant (minority) examples, then compare a perceptron trained directly on this imbalanced data against one trained on a subsampled version (Algorithm 11).

Because raw accuracy can be misleading on imbalanced data (predicting the majority class every time already scores well), we track **F1 on the positive class** as the more honest metric.

In [6]:
bc = load_breast_cancer()
Xb, yb_raw = bc.data, bc.target
minority_idx = np.where(yb_raw == 0)[0]
majority_idx = np.where(yb_raw == 1)[0]
keep_minority = RNG.choice(minority_idx, size=int(0.08 * len(minority_idx)), replace=False)
imb_idx = np.concatenate([keep_minority, majority_idx])
Xb_imb, yb_imb_raw = Xb[imb_idx], yb_raw[imb_idx]
yb_imb = np.where(yb_imb_raw == 0, 1, -1)

print(f"Imbalanced dataset: {np.sum(yb_imb == 1)} positive vs {np.sum(yb_imb == -1)} negative "
      f"({np.mean(yb_imb == 1) * 100:.1f}% positive)")

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb_imb, yb_imb, test_size=0.3, random_state=1, stratify=yb_imb
)
mu, sigma = Xb_train.mean(0), Xb_train.std(0)
sigma[sigma == 0] = 1
Xb_train_n = (Xb_train - mu) / sigma
Xb_test_n = (Xb_test - mu) / sigma

alpha = np.sum(yb_train == -1) / np.sum(yb_train == 1)
print(f"alpha (importance of positive class) = {alpha:.2f}")

p_plain = SimplePerceptron(max_iter=30, random_state=1).fit(Xb_train_n, yb_train)
pred_plain = p_plain.predict(Xb_test_n)

Xb_sub, yb_sub = subsample_map(Xb_train_n, yb_train, alpha, random_state=1)
p_sub = SimplePerceptron(max_iter=30, random_state=1).fit(Xb_sub, yb_sub)
pred_sub = p_sub.predict(Xb_test_n)

print(f"\n{'method':>18} | {'accuracy':>8} | {'F1 (positive class)':>20}")
print("-" * 52)
print(f"{'no subsampling':>18} | {accuracy(yb_test, pred_plain):>8.4f} "
      f"| {f1_score(yb_test, pred_plain, pos_label=1):>20.4f}")
print(f"{'subsampled':>18} | {accuracy(yb_test, pred_sub):>8.4f} "
      f"| {f1_score(yb_test, pred_sub, pos_label=1):>20.4f}")
n_pos_train = np.sum(yb_train == 1)
n_neg_kept = np.sum(yb_sub == -1)
print(f"\nTraining set sizes: full={len(yb_train)}  subsampled={len(yb_sub)} "
      f"({n_pos_train} positive + {n_neg_kept} negative kept)")

Imbalanced dataset: 16 positive vs 357 negative (4.3% positive)
alpha (importance of positive class) = 22.73

            method | accuracy |  F1 (positive class)
----------------------------------------------------
    no subsampling |   0.9911 |               0.9091
        subsampled |   0.7054 |               0.2326

Training set sizes: full=261  subsampled=28 (11 positive + 17 negative kept)


**Reading the result:** with `alpha` this large and the minority class already this small, subsampling shrinks the training set down to a tiny handful of points — too little for the perceptron to learn a good boundary from. This is a real, honest illustration of the book's own caveat (Section 5.1): subsampling "throws out a lot of data (especially for large alpha)." It works well when there's plenty of data to spare; it can backfire when the minority class is already scarce.

### Experiment B: One-vs-All and All-vs-All Multiclass (Section 5.2)

Now the multiclass case, using the 10-class Digits dataset. We train both from-scratch reductions (OVA with 10 binary classifiers, AVA with 45) and compare them against scikit-learn's `OneVsRestClassifier` and `OneVsOneClassifier`, both wrapping scikit-learn's own `Perceptron`.

In [7]:
digits = load_digits()
Xd, yd = digits.data, digits.target
print(f"Dataset shape: {Xd.shape[0]} examples, {Xd.shape[1]} features, {len(np.unique(yd))} classes")

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    Xd, yd, test_size=0.3, random_state=1, stratify=yd
)
mu_d, sigma_d = Xd_train.mean(0), Xd_train.std(0)
sigma_d[sigma_d == 0] = 1
Xd_train_n = (Xd_train - mu_d) / sigma_d
Xd_test_n = (Xd_test - mu_d) / sigma_d

base_fn = lambda: SimplePerceptron(max_iter=20, random_state=1)

print("\nTraining from-scratch OVA (10 binary classifiers)...")
ova = OVAClassifier(base_fn).fit(Xd_train_n, yd_train)
ova_acc = accuracy(yd_test, ova.predict(Xd_test_n))

print("Training from-scratch AVA (45 binary classifiers)...")
ava = AVAClassifier(base_fn).fit(Xd_train_n, yd_train)
ava_acc = accuracy(yd_test, ava.predict(Xd_test_n))

print("Training sklearn OneVsRestClassifier(Perceptron) for comparison...")
sk_ova = OneVsRestClassifier(SKPerceptron(max_iter=20, tol=None, random_state=1)).fit(Xd_train_n, yd_train)
sk_ova_acc = accuracy(yd_test, sk_ova.predict(Xd_test_n))

print("Training sklearn OneVsOneClassifier(Perceptron) for comparison...")
sk_ava = OneVsOneClassifier(SKPerceptron(max_iter=20, tol=None, random_state=1)).fit(Xd_train_n, yd_train)
sk_ava_acc = accuracy(yd_test, sk_ava.predict(Xd_test_n))

print(f"\n{'method':>28} | {'# binary classifiers':>21} | {'test accuracy':>13}")
print("-" * 68)
print(f"{'From-scratch OVA':>28} | {'10':>21} | {ova_acc:>13.4f}")
print(f"{'From-scratch AVA':>28} | {'45':>21} | {ava_acc:>13.4f}")
print(f"{'sklearn OneVsRest (OVA)':>28} | {'10':>21} | {sk_ova_acc:>13.4f}")
print(f"{'sklearn OneVsOne (AVA)':>28} | {'45':>21} | {sk_ava_acc:>13.4f}")

Dataset shape: 1797 examples, 64 features, 10 classes

Training from-scratch OVA (10 binary classifiers)...


Training from-scratch AVA (45 binary classifiers)...


Training sklearn OneVsRestClassifier(Perceptron) for comparison...
Training sklearn OneVsOneClassifier(Perceptron) for comparison...

                      method |  # binary classifiers | test accuracy
--------------------------------------------------------------------
            From-scratch OVA |                    10 |        0.9426
            From-scratch AVA |                    45 |        0.9667
     sklearn OneVsRest (OVA) |                    10 |        0.9315
      sklearn OneVsOne (AVA) |                    45 |        0.9593


**Reading the result:** as Theorems 3 & 4 in the book discuss, AVA trains `K(K-1)/2` classifiers instead of `K`, each on an easier binary sub-problem (only 2 classes' worth of data). This often — though not always — gives it an edge in accuracy over OVA, at a noticeably higher training cost.

## Use It

| API / Function | When to use it |
|---|---|
| `subsample_map(X, y, alpha)` | Imbalanced binary classification with a large-enough dataset that you can afford to drop majority-class examples |
| `OVAClassifier(base_learner_fn)` | Multiclass problems where training speed matters and you want linear scaling in the number of classes |
| `AVAClassifier(base_learner_fn)` | Multiclass problems where accuracy matters more than training time and the number of classes is modest |
| `sklearn.multiclass.OneVsRestClassifier` / `OneVsOneClassifier` | Production use — parallelizable, works with any sklearn-compatible binary estimator |

## Exercises

1. Implement the **oversampling** alternative to subsampling (replicate each positive example `alpha` times instead of dropping negatives) and re-run Experiment A — does it recover the F1 score that plain subsampling lost?
2. Extend `OVAClassifier` to break ties using a *confidence-weighted* vote instead of a raw `argmax`, and see if it changes accuracy on a dataset with more class overlap.
3. Time the training of OVA vs. AVA as the number of classes grows (e.g. on subsets of the Digits dataset with 3, 5, 7, 10 classes) and confirm the K vs. K(K-1)/2 scaling predicted by the book.

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Alpha-Weighted Classification** | "Just a fancy accuracy metric" | A binary classification variant where misclassifying a positive example costs `alpha` times as much as misclassifying a negative one |
| **Subsampling** | "Always improves imbalanced learning" | A reduction that discards a `1 - 1/alpha` fraction of the majority class; provably optimal in error *rate*, but can leave too little absolute data if the dataset is small |
| **One-versus-All (OVA)** | "The only way to do multiclass" | A reduction training one binary classifier per class (class vs. rest); linear in the number of classes but can be brittle to a single overconfident wrong classifier |
| **All-versus-All (AVA)** | "Just OVA done more times" | A reduction training one binary classifier per *pair* of classes; quadratic in the number of classes but often more accurate since each sub-problem is simpler |

## Summary

- Hard problems like imbalanced and multiclass classification can be **reduced** to plain binary classification, reusing any binary classifier you already have
- **Subsampling** turns an alpha-weighted problem into a plain one by discarding a `1 - 1/alpha` fraction of the majority class — powerful, but risky when data is already scarce
- **One-versus-All** trains `K` binary classifiers (linear cost, can be brittle)
- **All-versus-All** trains `K(K-1)/2` binary classifiers (quadratic cost, often more accurate on easier sub-problems)
- Reductions preserve theoretical error guarantees, but real datasets can still expose their practical trade-offs

---

**Next:** Chapter 6 — Linear Models